<a href="https://colab.research.google.com/github/Jaitech642/Binary-Search-Demonstrator/blob/main/Final_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import numpy as np
from datetime import datetime

# Define known parameters
url = 'https://web.archive.org/web/20230908091635/https://en.wikipedia.org/wiki/List_of_largest_banks'
exchange_rate_csv = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-PY0221EN-Coursera/labs/v2/exchange_rate.csv'
table_attribs = ['Name', 'MC_USD_Billion']
db_name = 'Banks.db'
table_name = 'Largest_banks'
output_csv_path = 'Largest_banks_data.csv'
log_file = 'code_log.txt'

def log_progress(message):
    ''' Logs the mentioned message at a given stage of the code execution to a log file. '''
    timestamp_format = '%Y-%m-%d-%H:%M:%S' # Year-Month-Day-Hour-Minute-Second
    now = datetime.now() # get current timestamp
    timestamp = now.strftime(timestamp_format)
    with open(log_file, "a") as f:
        f.write(timestamp + ' : ' + message + '\n')

def extract(url, table_attribs):
    ''' Extracts the required information from the website and saves it to a dataframe. '''
    page = requests.get(url).text
    data = BeautifulSoup(page, 'html.parser')
    df = pd.DataFrame(columns=table_attribs)

    # Extract the tables
    tables = data.find_all('tbody')

    # The 'By market capitalization' table is usually the first one (index 0) on this specific archived page
    rows = tables[0].find_all('tr')

    count = 0
    for row in rows:
        if count < 10:
            col = row.find_all('td')
            if len(col) != 0:
                # Extracting bank name from the second column (index 1) and market cap from the third column (index 2)
                # The market cap has a '\n' at the end which needs to be removed
                bank_name = col[1].find_all('a')[1]['title']
                market_cap = float(col[2].contents[0][:-1])

                # Append to dataframe
                data_dict = {"Name": bank_name, "MC_USD_Billion": market_cap}
                df1 = pd.DataFrame(data_dict, index=[0])
                df = pd.concat([df, df1], ignore_index=True)
                count += 1
        else:
            break

    return df

def transform(df, csv_path):
    ''' Transforms the dataframe by adding columns for GBP, EUR, and INR, rounded to 2 decimal places. '''
    # Read the exchange rate CSV into a dictionary
    exchange_rate = pd.read_csv(csv_path)
    # Convert dataframe to dictionary where Currency is key and Rate is value
    exchange_dict = exchange_rate.set_index('Currency').to_dict()['Rate']

    # Add the new columns
    df['MC_GBP_Billion'] = [np.round(x * exchange_dict['GBP'], 2) for x in df['MC_USD_Billion']]
    df['MC_EUR_Billion'] = [np.round(x * exchange_dict['EUR'], 2) for x in df['MC_USD_Billion']]
    df['MC_INR_Billion'] = [np.round(x * exchange_dict['INR'], 2) for x in df['MC_USD_Billion']]

    return df

def load_to_csv(df, output_path):
    ''' Saves the final dataframe as a CSV file. '''
    df.to_csv(output_path, index=False)

def load_to_db(df, sql_connection, table_name):
    ''' Saves the final dataframe to a database table. '''
    df.to_sql(table_name, sql_connection, if_exists='replace', index=False)

def run_query(query_statement, sql_connection):
    ''' Runs the stated query on the database table and prints the output. '''
    print(query_statement)
    query_output = pd.read_sql(query_statement, sql_connection)
    print(query_output)
    print("-" * 50)

# ==========================================
# Main Execution Flow
# ==========================================

if __name__ == '__main__':
    # 1. Initialize Log and Extract
    log_progress("Preliminaries complete. Initiating ETL process")
    df = extract(url, table_attribs)
    log_progress("Data extraction complete. Initiating Transformation process")

    # 2. Transform Data
    df = transform(df, exchange_rate_csv)
    log_progress("Data transformation complete. Initiating Loading process")

    # 3. Load to CSV
    load_to_csv(df, output_csv_path)
    log_progress("Data saved to CSV file")

    # 4. Connect to SQL DB and Load Data
    sql_connection = sqlite3.connect(db_name)
    log_progress("SQL Connection initiated")
    load_to_db(df, sql_connection, table_name)
    log_progress("Data loaded to Database as a table, Executing queries")

    # 5. Run Queries
    # a. Extract Name and MC_GBP_Billion for London office
    query_1 = f"SELECT Name, MC_GBP_Billion FROM {table_name}"
    run_query(query_1, sql_connection)

    # b. Extract Name and MC_EUR_Billion for Berlin office
    query_2 = f"SELECT Name, MC_EUR_Billion FROM {table_name} LIMIT 5"
    run_query(query_2, sql_connection)

    # c. Extract Name and MC_INR_Billion for New Delhi office
    query_3 = f"SELECT Name, MC_INR_Billion FROM {table_name} ORDER BY MC_INR_Billion DESC LIMIT 5"
    run_query(query_3, sql_connection)

    # 6. Close SQL Connection
    log_progress("Process Complete")
    sql_connection.close()
    log_progress("Server Connection closed")

/tmp/ipykernel_2688/3693825160.py:50: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, df1], ignore_index=True)


SELECT Name, MC_GBP_Billion FROM Largest_banks
                                      Name  MC_GBP_Billion
0                           JPMorgan Chase          346.34
1                          Bank of America          185.22
2  Industrial and Commercial Bank of China          155.65
3               Agricultural Bank of China          128.54
4                                HDFC Bank          126.33
5                              Wells Fargo          124.70
6                                     HSBC          119.12
7                           Morgan Stanley          112.66
8                  China Construction Bank          111.86
9                            Bank of China          109.45
--------------------------------------------------
SELECT Name, MC_EUR_Billion FROM Largest_banks LIMIT 5
                                      Name  MC_EUR_Billion
0                           JPMorgan Chase          402.62
1                          Bank of America          215.31
2  Industrial and Com